# Focus a _de novo_ model for Reinforcement Learning with the Reinvent prior


prior: Mol2Mol
using the required Files for this

### Imports

In [1]:
!which python
! conda activate reinvent4
import os
import shutil
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import reinvent
from reinvent.notebooks import load_tb_data, plot_scalars, get_image, create_mol_grid
from reinvent.scoring.transforms import ReverseSigmoid
from reinvent.scoring.transforms.sigmoids import Parameters as SigmoidParameters

import ipywidgets as widgets

%load_ext tensorboard

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
/home/a/miniconda3/envs/reinvent4/bin/python
/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)

CondaError: Run 'conda init' before 'conda activate'



### WD Setup

In [2]:
top = os.path.abspath("./..")
wd = f'{top}/tmp/R4_notebooks_output'

top
wd

'/home/a/REINVENT4/ReinventStudies/tmp/R4_notebooks_output'

In [3]:
if not os.path.isdir(wd):
    shutil.rmtree(wd, ignore_errors=True)
    os.mkdir(wd)

os.chdir(wd)
wd

'/home/a/REINVENT4/ReinventStudies/tmp/R4_notebooks_output'

## Reinvent Toml Setup


In [4]:

stage1_checkpoint = "stage1.chkpt"

max_score = 0.8  # terminate if this total score is exceeded
min_steps = 2  # run for at least this number of steps
max_steps = 3  # terminate entire run when exceeded
batch_size = 4

inp_smi = os.path.abspath(os.path.join(top, "mols", "Rad51","cam833.smi"))
prior_filename = os.path.abspath(os.path.join(top, "priors", "mol2mol_similarity.prior"))
agent_filename = prior_filename

maize_executable = "/home/a/miniconda3/envs/maize-dev/bin/python"
maize_config = os.path.abspath(os.path.join(top, "configs","Maize","maize-mol2mol-config.toml"))
maize_workflow = os.path.abspath(os.path.join(top, "run_maize_mol2mol.py"))


In [5]:
stage1_parameters=f"""run_type = "staged_learning"
device = "cuda"  # set torch device e.g. "cpu"
tb_logdir = "tb_logs"  # name of the TensorBoard logging directory
json_out_config = "_staged_learning.json"  # write this TOML to JSON

# [responder]
# endpoint = "http://localhost:8080/api/report"
# frequency = 1  # Report every 10 steps

[parameters]
summary_csv_prefix = "staged_learning"  # prefix for the CSV file
use_checkpoint = true  # if true read diversity filter from agent_file
purge_memories = false  # if true purge all diversity filter memories after each stage

## Mol2Mol
prior_file = "{prior_filename}"
agent_file = "{agent_filename}"
smiles_file = "{inp_smi}"
sample_strategy = "beamsearch"  # multinomial or beamsearch (deterministic)
distance_threshold = 100


batch_size = {batch_size}          # network
unique_sequences = true  # if true remove all duplicates raw sequences in each step
                         # only here for backward compatibility
randomize_smiles = true  # if true shuffle atoms in SMILES randomly
tb_isim = true  # track iSIM similarity in TensorBoard

[learning_strategy]

type = "dap"      # dap: only one supported
sigma = 128       # sigma of the RL reward function
rate = 0.0001     # for torch.optim


[diversity_filter]  # optional, comment section out or remove if unneeded
                    # NOTE: also memorizes all seen SMILES

type = "IdenticalMurckoScaffold" # IdenticalTopologicalScaffold,
                                 # ScaffoldSimilarity, PenalizeSameSmiles
bucket_size = 25                 # memory size in number of compounds
minscore = 0.4                   # only memorize if this threshold is exceeded
minsimilarity = 0.4              # minimum similarity for ScaffoldSimilarity
penalty_multiplier = 0.5         # penalty factor for PenalizeSameSmiles

[[stage]]



termination = "simple"  # termination criterion fot this stage
max_score = {max_score}  # terminate if this total score is exceeded
min_steps = {min_steps}  # run for at least this number of steps
max_steps = {max_steps}  # terminate entire run when exceeded


chkpt_file = "{stage1_checkpoint}"  # name of the checkpoint file, can be reused as agent

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]

[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"

params.smarts = [
    "[*;r8]",
    "[*;r9]",
    "[*;r10]",
    "[*;r11]",
    "[*;r12]",
    "[*;r13]",
    "[*;r14]",
    "[*;r15]",
    "[*;r16]",
    "[*;r17]",
    "[#8][#8]",
    "[#6;+]",
    "[#16][#16]",
    "[#7;!n][S;!$(S(=O)=O)]",
    "[#7;!n][#7;!n]",
    "C#C",
    "C(=[O,S])[O,S]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#7;!n]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#16;!s][C;!$(C(=[O,N])[N,O])][#16;!s]"
]

[[stage.scoring.component]]
[stage.scoring.component.QED]

[[stage.scoring.component.QED.endpoint]]
name = "QED"
weight = 0.6

[[stage.scoring.component]]
[stage.scoring.component.MAIZE]
[[stage.scoring.component.MAIZE.endpoint]]
weight = 1.0  # user chosen name for output
name = "DockingScores"

# Score transformation settings - lower is better in this case
transform.type = "reverse_sigmoid"
transform.high = -3.0
transform.low = -15.0
transform.k = 0.5


params.executable = "{maize_executable}"
params.workflow = "{maize_workflow}"
# params.config = "{maize_config}"  # optional
params.debug = true
params.keep = true
params.distance_threshold =100
params.smiles_file = "{inp_smi}"
params.sample_strategy = "beamsearch"  # multinomial or beamsearch (deterministic)
"""


In [6]:
stage1_config_filename = "stage1.toml"

with open(stage1_config_filename, "w") as tf:
    tf.write(stage1_parameters)

# Run Reinvent4


In [7]:
shutil.rmtree("tb_logs_0", ignore_errors=True)

%time
!reinvent -l stage1.log $stage1_config_filename




CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 5.25 μs
/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)


In [8]:
! echo $wd/tb-logs_0
%tensorboard --bind_all --logdir $wd/tb_logs_0 --port=8008


/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
/home/a/REINVENT4/ReinventStudies/tmp/R4_notebooks_output/tb-logs_0


Reusing TensorBoard on port 8008 (pid 4902), started 1:10:37 ago. (Use '!kill 4902' to kill it.)